# 06 — Regresión logística (Persona B)

**Objetivo:** primer modelo con información del partido. Comparamos en **validación (2023-24)**:
- **a.** logística **solo con `elo_diff`**, y
- **b.** logística con **todas las `FEATURE_COLS`** (forma, Elo y descanso),

contra el baseline de frecuencias y las cuotas del mercado (notebook `01_baseline_B`).
El conjunto de prueba (2024-25) no se toca.

## 1. Carga, corte temporal y partidos evaluables
- `split(df)` sin `use_test`: train = 2017-18 a 2022-23, validación = 2023-24.
- Train se **ordena por fecha**, porque `TimeSeriesSplit` asume que las filas están en orden temporal.
- Igual que en el notebook 01, todo se evalúa sobre **los mismos partidos que tienen cuotas**.

In [ ]:
import sys; sys.path.append("..")
import numpy as np
import pandas as pd

from src.splits import split
from src.features.build import FEATURE_COLS
from src.models.baseline import FrequencyBaseline
from src.models.logistic import tune_logistic, coef_table
from src.evaluation.metrics import evaluate, compare, confusion, implied_probs, to_hda
from src.evaluation.registro import guardar_resultado

df = pd.read_csv("../data/processed/features.csv", parse_dates=["date"])
train, val = split(df)
train = train.sort_values(["date", "match_id"])
assert train["date"].max() < val["date"].min()

odds_cols = ["odds_H", "odds_D", "odds_A"]
val_ok = val[val[odds_cols].notna().all(axis=1)]
y = val_ok["result"]
print(f"train: {len(train)} partidos | val: {len(val)} | val con cuotas: {len(val_ok)}")
print("NaN por feature en train:")
print(train[FEATURE_COLS].isna().sum()[lambda s: s > 0].to_string())

Los NaN vienen del **primer partido de cada equipo** en el histórico (no hay forma ni descanso previos).
No eliminamos esas filas: se imputan con la mediana.

## 2. Por qué todo va dentro de un `Pipeline`
`SimpleImputer(median) → StandardScaler → LogisticRegression`

La mediana del imputador y la media/desviación del escalador son **parámetros aprendidos de los datos**.
Si se calcularan con todo el dataset antes de separar, información de validación (o del futuro, dentro de
cada pliegue de la validación cruzada) se filtraría al entrenamiento. Dentro del `Pipeline`, cada `fit`
(de un pliegue o de train completo) recalcula esos valores **solo con sus propias filas**, y `predict`
únicamente los aplica.

Escalar además importa por dos razones: la penalización L2 (controlada por `C`) trata a todas las variables
por igual solo si están en la misma escala, y así los coeficientes quedan comparables (efecto de **una
desviación estándar** de cada variable).

## 3. Ajuste de `C` con `TimeSeriesSplit` dentro de train
`C` es la inversa de la regularización: `C` pequeño = coeficientes más encogidos hacia 0.
`tune_logistic()` prueba una grilla de `C` con `TimeSeriesSplit(n_splits=5)` (cada pliegue entrena con el
pasado y valida con el bloque siguiente) y `scoring="neg_log_loss"`; luego reajusta el mejor con todo train.
**Validación no participa en esta elección.**

In [ ]:
modelos = {"logistica_elo": ["elo_diff"], "logistica_todas": FEATURE_COLS}
busquedas = {nombre: tune_logistic(train, cols) for nombre, cols in modelos.items()}

cv = pd.DataFrame({nombre: -s.cv_results_["mean_test_score"] for nombre, s in busquedas.items()},
                  index=pd.Index(busquedas["logistica_elo"].cv_results_["param_logistica__C"], name="C"))
for nombre, s in busquedas.items():
    print(f"{nombre}: mejor C = {s.best_params_['logistica__C']:.4g} | log loss CV = {-s.best_score_:.4f}")
cv.round(4)

Log loss medio en los 5 pliegues temporales de train (menor es mejor). Con `C` muy pequeño ambos modelos
se acercan al baseline (~1.05); a partir de `C ≈ 0.01` la curva es casi plana.

## 4. Comparación en validación
Probabilidades con `to_hda()` (el `predict_proba` de sklearn viene en orden alfabético A, D, H) y métricas
con `evaluate()`. Baseline y cuotas se recalculan sobre los mismos partidos para mostrar la tabla completa.

In [ ]:
probs = {
    "baseline_frecuencias": FrequencyBaseline().fit(None, train["result"]).predict_proba(val_ok),
    "cuotas_mercado": implied_probs(val_ok),
}
for nombre, cols in modelos.items():
    m = busquedas[nombre].best_estimator_
    probs[nombre] = to_hda(m, m.predict_proba(val_ok[cols]))

resultados = {nombre: evaluate(y, p, nombre) for nombre, p in probs.items()}
compare(list(resultados.values()))

### Guardar las dos logísticas en la tabla compartida
`guardar_resultado()` reemplaza la fila si el modelo ya existe, así que se puede re-ejecutar sin duplicar.

In [ ]:
guardar_resultado(resultados["logistica_elo"])
tabla = guardar_resultado(resultados["logistica_todas"])
tabla.sort_values("log_loss").reset_index(drop=True)

## 5. ¿Las diferencias son reales o ruido?
Con 380 partidos la log loss tiene bastante varianza. Hacemos un **bootstrap pareado**: remuestreamos los
mismos partidos para ambos modelos y miramos la distribución de la diferencia de log loss
(negativo = el primer modelo es mejor). Cada métrica sigue pasando por `evaluate()`.

In [ ]:
rng = np.random.default_rng(42)
idx_boot = [rng.integers(0, len(y), len(y)) for _ in range(1000)]
y_arr = y.to_numpy()

def dif_bootstrap(a, b):
    d = np.array([evaluate(y_arr[i], probs[a][i])["log_loss"] - evaluate(y_arr[i], probs[b][i])["log_loss"]
                  for i in idx_boot])
    return {"comparación": f"{a} − {b}",
            "dif. observada": resultados[a]["log_loss"] - resultados[b]["log_loss"],
            "IC 95% inf": np.percentile(d, 2.5), "IC 95% sup": np.percentile(d, 97.5),
            "% remuestreos con el primero mejor": (d < 0).mean()}

pd.DataFrame([dif_bootstrap("logistica_todas", "logistica_elo"),
              dif_bootstrap("logistica_elo", "baseline_frecuencias"),
              dif_bootstrap("logistica_todas", "cuotas_mercado")]).round(4)

## 6. Coeficientes por clase
La logística multinomial tiene **un vector de coeficientes por clase**; la probabilidad de cada resultado es
el *softmax* de los tres puntajes lineales. Como las variables están estandarizadas, cada coeficiente es el
cambio en el puntaje (log-odds) de esa clase cuando la variable sube **una desviación estándar**.
Con penalización L2, los tres coeficientes de cada variable suman 0, así que solo importan las diferencias
entre clases. La columna **H − A** es el cambio en el log-odds de *gana local vs gana visitante*, y
`exp(H − A)` el factor por el que se multiplican esas odds.

### a. Solo `elo_diff`

In [ ]:
def tabla_coef(nombre):
    t = coef_table(busquedas[nombre].best_estimator_, modelos[nombre])
    t["H − A"] = t["H"] - t["A"]
    t["exp(H − A)"] = np.exp(t["H − A"])
    return t.reindex(t["H − A"].abs().sort_values(ascending=False).index)

print(f"1 desviación estándar de elo_diff en train ≈ {train['elo_diff'].std():.0f} puntos Elo")
tabla_coef("logistica_elo").round(3)

### b. Todas las features (ordenadas por |H − A|)

In [ ]:
tabla_coef("logistica_todas").round(3)

**Ojo al interpretar la tabla b:** `elo_diff = home_elo + 60 − away_elo` es una combinación lineal exacta de
otras dos columnas (y `rest_diff = home_rest − away_rest` también). Con variables colineales la regularización
**reparte** el efecto entre ellas, así que el coeficiente individual de `Elo del local` o de `Diferencia de Elo`
no es el efecto "puro" de esa variable: el efecto total del Elo es la suma de las tres. Para predecir no es un
problema; para explicar conviene mirar el modelo **a** o agrupar las tres columnas de Elo.

## 7. Matriz de confusión (logística con todas las features)

In [ ]:
pred = pd.Series(np.array(["H", "D", "A"])[probs["logistica_todas"].argmax(axis=1)])
print("Predicciones por clase:", pred.value_counts().to_dict())
print(f"Probabilidad media de empate: {probs['logistica_todas'][:, 1].mean():.3f} "
      f"(máxima: {probs['logistica_todas'][:, 1].max():.3f})")
confusion(y, probs["logistica_todas"])

## 8. Conclusión

| modelo (validación 2023-24, 380 partidos) | log loss | Brier | accuracy |
|---|---|---|---|
| cuotas del mercado | **0.909** | **0.533** | **58.9 %** |
| logística con todas las features | 0.928 | 0.546 | 58.7 % |
| logística solo `elo_diff` | 0.932 | 0.549 | 57.4 % |
| baseline de frecuencias | 1.054 | 0.637 | 46.1 % |

1. **La tarea 5 se cumple:** las dos logísticas mejoran claramente al baseline (−0.12 de log loss; el IC 95 %
   del bootstrap, [−0.160, −0.082], queda entero por debajo de 0).
2. **El Elo concentra casi toda la señal.** Agregar forma y descanso baja la log loss solo 0.004 en validación,
   y el IC 95 % de esa diferencia, [−0.013, +0.005], incluye el 0. Además, en la validación cruzada dentro de
   train el modelo con todas las features fue *peor* (0.982 frente a 0.975). Conclusión honesta: las features
   extra aportan, como mucho, una mejora marginal que no podemos distinguir del ruido con una temporada.
   El descanso no aporta nada (coeficientes ≈ 0).
3. **El mercado sigue siendo mejor** por 0.019 de log loss (IC 95 % [0.002, 0.036]): una distancia pequeña,
   pero real.
4. **Sin señales de fuga:** accuracy < 60 %. La log loss de validación (0.93) es menor que la de la validación
   cruzada (≈0.975), pero las cuotas también bajan a 0.909 en esa temporada: 2023-24 fue simplemente más
   predecible.
5. **Nunca predice empate** (probabilidad máxima de D = 0.32). Es normal en este problema, y por eso juzgamos
   con log loss y Brier y no con accuracy.
6. **Interpretación:** en el modelo solo con Elo, +1 desviación estándar de `elo_diff` (≈153 puntos) multiplica
   por ≈2.8 las odds de *gana local* frente a *gana visitante*. En el modelo completo ese efecto se reparte entre
   `elo_diff`, `home_elo` y `away_elo` porque son colineales.

**Siguientes pasos:** Random Forest y XGBoost (tarea 7), para ver si hay no linealidades que la logística no
captura. Para explicar el modelo (SHAP, tarea 12) conviene proponer a Persona A dejar solo una representación
del Elo (`elo_diff` o el par `home_elo`/`away_elo`).
